In [ ]:
import random

# --- 1. Simulate (or load) raw sensor data ---


def generate_raw_sensor_data(n_samples):
    """Simulate raw sensor data as list of dicts."""
    data = []
    for i in range(n_samples):
        # Example: sensor reports an integer and a float value
        data.append({
            'id': i,
            'raw_int': random.randint(0, 100),
            'raw_float': round(random.uniform(-10.0, 10.0), 2),
        })
    return data

# --- 2. The "ground-truth" conversion ---


def gold_standard_conversion(raw):
    """Gold-standard conversion to a ROS message dict."""
    # In real application, this should match the specification
    msg = {
        'seq': raw['id'],
        'value_int': raw['raw_int'],
        'value_float': raw['raw_float'],
    }
    return msg

# --- 3. The conversion under test (replace with yours) ---


def your_conversion_function(raw):
    """Your converter, which may have bugs."""
    # For demo, intentionally make a 1% error rate
    msg = {
        'seq': raw['id'],
        'value_int': raw['raw_int'],
        'value_float': raw['raw_float'],
    }
    # Introduce occasional error
    if random.random() < 0.01:
        msg['value_float'] += random.uniform(0.1, 1.0)
    return msg

# --- 4. Compare function with optional float tolerance ---


def messages_equal(msg1, msg2, float_tol=1e-4):
    for k in msg1:
        if k not in msg2:
            return False
        v1 = msg1[k]
        v2 = msg2[k]
        if isinstance(v1, float):
            if abs(v1 - v2) > float_tol:
                return False
        else:
            if v1 != v2:
                return False
    return True

# --- 5. KPI validation routine ---


def validate_kpi(raw_data, ground_truth_func, convert_func, float_tol=1e-4):
    num_total = len(raw_data)
    num_correct = 0
    errors = []
    for i, raw in enumerate(raw_data):
        exp_msg = ground_truth_func(raw)
        got_msg = convert_func(raw)
        if messages_equal(exp_msg, got_msg, float_tol):
            num_correct += 1
        else:
            errors.append((i, raw, exp_msg, got_msg))
    accuracy = (num_correct / num_total) * 100.0
    return accuracy, errors

In [3]:
N = 10000
raw_data = generate_raw_sensor_data(N)

accuracy, errors = validate_kpi(
    raw_data,
    gold_standard_conversion,
    your_conversion_function,
    float_tol=1e-3
)

print(f"KPI validation completed on {N} samples.")
print(f"Accuracy: {accuracy:.2f}%")
if accuracy >= 99.0:
    print("✅ KPI PASSED")
else:
    print("❌ KPI FAILED")

print(f"Number of failed conversions: {len(errors)}")
if errors:
    # Print a sample error (field mismatch example)
    idx, raw, exp, got = errors[0]
    print(f"\nSample failure at index {idx}:")
    print("  Input:            ", raw)
    print("  Expected message: ", exp)
    print("  Got message:      ", got)

KPI validation completed on 10000 samples.
Accuracy: 98.92%
❌ KPI FAILED
Number of failed conversions: 108

Sample failure at index 170:
  Input:             {'id': 170, 'raw_int': 36, 'raw_float': -1.45}
  Expected message:  {'seq': 170, 'value_int': 36, 'value_float': -1.45}
  Got message:       {'seq': 170, 'value_int': 36, 'value_float': -0.61395866163774}
